# Fine-tune Granite Speech for Italian ASR

A short tutorial for fine-tuning Granite Speech on Italian ASR with `espnet/yodas-granary`.


In [1]:
# !pip install -q git+https://github.com/huggingface/transformers datasets[audio] accelerate evaluate jiwer soundfile torchaudio

## Load Dataset

In [2]:
from itertools import islice

from datasets import Audio, Dataset, DatasetDict, load_dataset

stream = load_dataset("espnet/yodas-granary", "Italian", split="asr_only", streaming=True)
rows = list(islice(stream, 5400))

dataset = Dataset.from_list(rows)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

dataset = DatasetDict(
    {
        "train": dataset.select(range(5000)),
        "validation": dataset.select(range(5000, 5200)),
        "test": dataset.select(range(5200, 5400)),
    }
)

dataset

/opt/conda/envs/test_notebook/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['utt_id', 'audio', 'duration', 'lang', 'task', 'text', 'translation_en', 'original_audio_id', 'original_audio_offset'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['utt_id', 'audio', 'duration', 'lang', 'task', 'text', 'translation_en', 'original_audio_id', 'original_audio_offset'],
        num_rows: 200
    })
    test: Dataset({
        features: ['utt_id', 'audio', 'duration', 'lang', 'task', 'text', 'translation_en', 'original_audio_id', 'original_audio_offset'],
        num_rows: 200
    })
})

## Load Processor And Model

In [3]:
import torch
import torch.nn as nn
from transformers.models.granite_speech import GraniteSpeechForConditionalGeneration, GraniteSpeechProcessor

model_name = "ibm-granite/granite-4.0-1b-speech"
processor = GraniteSpeechProcessor.from_pretrained(model_name)
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
device_map = {"": 0} if torch.cuda.is_available() else None

model = GraniteSpeechForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=compute_dtype,
    device_map=device_map,
)

for name, param in model.named_parameters():
    param.requires_grad = "projector" in name

model.config.use_cache = False
trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
all_params = sum(param.numel() for param in model.parameters())
print(f"Trainable params: {trainable_params:,} / {all_params:,} ({100 * trainable_params / all_params:.2f}%)")

Loading weights: 100%|██████████| 954/954 [00:01<00:00, 709.83it/s] 


Trainable params: 35,697,664 / 2,313,141,596 (1.54%)


## Data preprocessing

Let's continue with data processing:

- The text format requires some preprocessing. (e.g. replace <COMMA> with ,)
- Add an instruction prompt (e.g. Can you transcribe the following speech<|audio|>?)
- Filter non-verbal examples (e.g. <noise>)

In [4]:
def process_gigaspeech_transcript(text):
    text = text.replace(" <COMMA>", ",")
    text = text.replace(" <PERIOD>", ".")
    text = text.replace(" <QUESTIONMARK>", "?")
    text = text.replace(" <EXCLAMATIONPOINT>", "!")
    text = text.lower()
    return text

def prep_example(example, tokenizer):
    instruction = "Please transcribe the following audio to text<|audio|>"
    chat = [dict(role="user", content=instruction)]
    example["prompt"] = tokenizer.apply_chat_template(
        chat,
        add_generation_prompt=True,
        tokenize=False,
    )
    example["text"] = process_gigaspeech_transcript(example["text"])
    return example

def prepare_dataset(ds, processor):
    columns_to_remove = [col for col in ds.column_names if col not in ["audio", "text"]]
    ds = ds.cast_column("audio", Audio(sampling_rate=processor.audio_processor.sampling_rate))
    ds = ds.map(prep_example,
        fn_kwargs=dict(tokenizer=processor.tokenizer),
        remove_columns=columns_to_remove,
    )
    ds = ds.filter(lambda x: x["text"] not in ["<other>", "<noise>", "<music>", "<sil>"])
    return ds

In [5]:
train_dataset = prepare_dataset(dataset["train"], processor)
val_dataset = prepare_dataset(dataset["validation"], processor)
test_dataset = prepare_dataset(dataset["test"], processor)

Filter:  40%|████      | 2000/5000 [00:12<00:18, 159.89 examples/s]

Filter: 100%|██████████| 200/200 [00:01<00:00, 156.32 examples/s]


In [6]:
from IPython.display import Audio as IPAudio
audio = train_dataset[0]["audio"]
print(train_dataset[0]["text"])
IPAudio(data=audio["array"], rate=audio["sampling_rate"])

oggi parliamo di...


## Running inference + WER computation
Now let's compute word error rate, for that we'll need to define a collator, which will also be used for finetuning

In [7]:
from contextlib import nullcontext

import evaluate
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer
from transformers.feature_extraction_utils import BatchFeature
from torch.utils.data import DataLoader
import tqdm


def extract_audio_array(audio):
    """Handle both legacy dict format and new torchcodec AudioDecoder objects."""
    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        return samples.data.squeeze(0).numpy()
    elif isinstance(audio, dict):
        return audio["array"]
    return audio


class GraniteCollator:
    def __init__(self, processor, inference_mode=False):
        self.processor = processor
        self.inference_mode = inference_mode

    def __call__(self, examples):
        prompts = [example["prompt"] for example in examples]
        audios = [extract_audio_array(example["audio"]) for example in examples]

        processed = self.processor(prompts, audios, return_tensors="pt", padding=True, padding_side="left")
        input_ids = processed.input_ids
        attention_mask = processed.attention_mask
        labels = None
        # tokenize targets
        if not self.inference_mode:
            targets = [example["text"] + self.processor.tokenizer.eos_token for example in examples]
            targets = self.processor.tokenizer(targets, return_tensors="pt", padding=True, padding_side="right")
            # combine prompt+targets
            input_ids = torch.cat([input_ids, targets.input_ids], dim=1)
            attention_mask = torch.cat([attention_mask, targets.attention_mask], dim=1)
            labels = targets.input_ids.clone()
            # Set non-target tokens to -100 for loss calculation
            labels[~(targets.attention_mask.bool())] = -100  
            labels = torch.cat([torch.full_like(processed.input_ids, -100), labels], dim=1)

        return BatchFeature(data={
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "input_features": processed.input_features,
            "input_features_mask": processed.input_features_mask
        })

def compute_wer(model, processor, cur_dataset, batch_size=1):
    collator = GraniteCollator(processor, inference_mode=True)
    dataloader = DataLoader(cur_dataset, batch_size=batch_size, collate_fn=collator, num_workers=8)
    normalizer = EnglishTextNormalizer({})
    wer_metric = evaluate.load("wer")
    device = next(model.parameters()).device
    model = model.eval()
    autocast_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    
    all_outputs = []
    for batch in tqdm.tqdm(dataloader, desc="Running inference"):
        batch = batch.to(device)
        autocast_context = torch.amp.autocast("cuda", dtype=autocast_dtype) if getattr(device, "type", str(device)) == "cuda" else nullcontext()
        with torch.inference_mode(), autocast_context:
            outputs = model.generate(**batch, max_new_tokens=128, num_beams=1)
        input_length = batch.input_ids.shape[1]
        outputs = outputs[:, input_length:].cpu()
        for x in outputs:
            all_outputs.append(processor.tokenizer.decode(x, skip_special_tokens=True))
        
    gt_texts = [normalizer(x) for x in cur_dataset["text"]]
    all_outputs = [normalizer(x) for x in all_outputs]
    wer = wer_metric.compute(references=gt_texts, predictions=all_outputs)
    return wer

In [8]:
wer_before_train = compute_wer(model, processor, test_dataset)
print(f"WER before finetuning {wer_before_train*100:.3f}")

Running inference:  18%|█▊        | 36/200 [00:35<01:26,  1.89it/s]

Running inference: 100%|██████████| 200/200 [03:15<00:00,  1.02it/s]


WER before finetuning 70.540


# Finetuning Granite Speech
Let's finetune the model on our small training set. We'll only tune the LoRA adapters and the projector, to speed up training and avoid overfitting.

In [9]:
from transformers import TrainingArguments, Trainer

torch.cuda.empty_cache()

args = TrainingArguments(
    output_dir="save_dir",
    remove_unused_columns=False,
    report_to="none",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    eval_strategy="no",
    save_strategy="no",
    optim="paged_adamw_8bit",
    dataloader_num_workers=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1.0,
    warmup_steps=50,
    logging_steps=10,
    learning_rate=3e-5,
    data_seed=42,
    gradient_checkpointing=False,
)
data_collator = GraniteCollator(processor)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=processor,
)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 100257, 'bos_token_id': 100257, 'pad_token_id': 100256}.


Step,Training Loss
10,9.840435
20,9.206676
30,9.016418
40,8.128776
50,8.541417
60,9.010206
70,8.058769
80,9.155116
90,8.691121
100,7.778294


TrainOutput(global_step=625, training_loss=7.319370867919922, metrics={'train_runtime': 604.1399, 'train_samples_per_second': 8.276, 'train_steps_per_second': 1.035, 'total_flos': 1.1747686827889344e+16, 'train_loss': 7.319370867919922, 'epoch': 1.0})

## Checking for improvements
Looks like both the training and validation loss are dropping. Let's check if the test WER improved by our very lightweight finetuning.

In [10]:
wer_after_train = compute_wer(model, processor, test_dataset)

print(f"WER after finetuning {wer_after_train*100:.3f}")
if "wer_before_train" in globals():
    print(f"WER improvement {(wer_before_train - wer_after_train)*100:.3f}")

Running inference:   8%|▊         | 15/200 [00:13<02:58,  1.04it/s]

Running inference: 100%|██████████| 200/200 [03:23<00:00,  1.02s/it]


WER after finetuning 58.216
WER improvement 12.324
